# Exploração e coleta dos dados de votações da Câmara

Este notebook documenta a exploração inicial dos dados abertos da Câmara dos Deputados
usados no projeto O Gabinete. Aqui baixamos os arquivos brutos, inspecionamos sua estrutura
e geramos as versões limpas usadas no cálculo de similaridade entre deputados.

A lógica final de produção está nos arquivos `coleta.py` e `processamento.py`, dentro de
`pipeline/`. Este notebook serve como registro exploratório e visual do processo.


In [2]:
import requests
import pandas as pd
import os

## Definindo o período e as fontes de dados

Como o grupo ainda não fechou oficialmente o período de análise, usamos 2024 como
referência provisória (item #2 do board).

Duas fontes são usadas:
- `votacoesVotos`: o voto individual de cada deputado em cada votação
- `votacoes`: metadados sobre cada votação (usado depois para decidir filtros)

In [3]:
ANO_REFERENCIA = 2024

url_votos = f'https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-{ANO_REFERENCIA}.csv'
url_votacoes = f'https://dadosabertos.camara.leg.br/arquivos/votacoes/csv/votacoes-{ANO_REFERENCIA}.csv'

## Baixando os dados brutos

Verifica se os arquivos já existem em `dados/brutos/` antes de baixar, para não repetir
o download toda vez que o notebook rodar. Os caminhos são calculados a partir da raiz do
projeto (via `encontrar_raiz`), então funciona independente de onde o notebook estiver salvo.

In [ ]:
from pathlib import Path

def encontrar_raiz(marcador="requirements.txt"):
    caminho = Path.cwd()
    while not (caminho / marcador).exists():
        caminho = caminho.parent
    return caminho

RAIZ = encontrar_raiz()
pasta = RAIZ / "dados" / "brutos"
pasta.mkdir(parents=True, exist_ok=True)

for url in [url_votos, url_votacoes]:
    nome_arquivo = url.split('/')[-1]
    caminho_arquivo = pasta / nome_arquivo

    if not caminho_arquivo.exists():
        print(f'Baixando {nome_arquivo}...')
        response = requests.get(url)
        with open(caminho_arquivo, 'wb') as f:
            f.write(response.content)
    else:
        print(f'{nome_arquivo} já existe. Pulando download.')

with open(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', 'r', encoding='utf-8') as f:
    print(f.readline())

votacoesVotos-2024.csv já existe. Pulando download.
votacoes-2024.csv já existe. Pulando download.
﻿"idVotacao";"uriVotacao";"dataHoraVoto";"voto";"deputado_id";"deputado_uri";"deputado_nome";"deputado_siglaPartido";"deputado_uriPartido";"deputado_siglaUf";"deputado_idLegislatura";"deputado_urlFoto"



### Observação sobre o formato do CSV

O cabeçalho impresso acima revela dois detalhes importantes do arquivo da Câmara:
- separador é `;`, não vírgula (por isso o `sep=';'` na leitura)
- o arquivo tem um BOM no início (por isso o `encoding='utf-8-sig'`, que remove esse
  caractere invisível automaticamente)

## Lendo o CSV de votos para o pandas

In [10]:
df = pd.read_csv(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

## Limpando o arquivo de votos

O arquivo bruto traz dados do deputado (nome, partido, UF, foto) repetidos em toda linha,
o que é redundante e deixa o arquivo maior do que precisa. Mantemos só o essencial para o
cálculo de similaridade: `idVotacao`, `deputado_id` e `voto`, além de `siglaPartido` e
`siglaUf` (usados depois para colorir os vértices do grafo).

O resultado é salvo em `dados/processado/votos-{ano}-limpo.csv`.

In [11]:
colunas_uteis = ['idVotacao', 'deputado_id', 'voto', 'deputado_siglaPartido', 'deputado_siglaUf']
df_limpo = df[colunas_uteis]

(RAIZ / 'dados' / 'processado').mkdir(parents=True, exist_ok=True)
df_limpo.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-limpo.csv', index=False)


## Gerando a tabela de deputados

Como os dados do deputado se repetem em cada linha de voto, extraímos um registro único
por `deputado_id` a partir do próprio arquivo bruto, incluindo a URL da foto (que já vem
válida no CSV, sem precisar de chamada extra à API).

Resultado salvo em `dados/processado/deputados.csv`.

In [12]:
deputados_df = df.drop_duplicates(subset='deputado_id')[
    ['deputado_id', 'deputado_nome', 'deputado_siglaPartido', 'deputado_siglaUf', 'deputado_urlFoto']
].copy()

deputados_df.to_csv(RAIZ / 'dados' / 'processado' / 'deputados.csv', index=False)